# Entanglement — Amazon Braket

Entanglement is a uniquely quantum correlation: measuring one qubit
instantly determines the state of its partner.  We create the Bell
state $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ and
verify perfect correlations.

In [ ]:
import json

from braket.circuit import Circuit
from braket.devices import LocalSimulator

In [ ]:
device = LocalSimulator()

## Bell state |\u03a6+\rangle = (|00\rangle + |11\rangle)/\u221a2

H on qubit 0 followed by CNOT creates the maximally entangled Bell pair.

In [ ]:
circuit = Circuit()
circuit.h(0)
circuit.cnot(0, 1)
print(circuit)

result = device.run(circuit, shots=0).result()
print(f"amplitudes: {result.result_types[0].value}")
print(f"probabilities: {result.result_types[1].value}")

## Perfect correlations

Only |00\rangle and |11\rangle appear — the qubits always agree.

In [ ]:
measured = Circuit()
measured.add_circuit(circuit)
measured.measure(0)
measured.measure(1)

result = device.run(measured, shots=4000).result()
counts = result.result_types[0].value
print(f"counts: {counts}")
agree = counts.get("00", 0) + counts.get("11", 0)
print(f"fraction same: {agree / 4000:.1%}")

## Measure qubit 0 only

After measuring qubit 0, qubit 1 collapses to the same value.

In [ ]:
circuit = Circuit()
circuit.h(0)
circuit.cnot(0, 1)
circuit.measure(0)

result = device.run(circuit, shots=2000).result()
counts = result.result_types[0].value
print(f"counts (qubit 0 measured): {counts}")
print("After measuring qubit 0, qubit 1 is always the same value.")

## Four Bell states

All four maximally entangled two-qubit states:

In [ ]:
bell_states = {
    "|Phi+>": {"swap": False, "phase_x": True},
    "|Phi->": {"swap": False, "phase_x": False},
    "|Psi+>": {"swap": True, "phase_x": True},
    "|Psi->": {"swap": True, "phase_x": False},
}

for name, params in bell_states.items():
    circuit = Circuit()
    if params["phase_x"]:
        circuit.x(0)
    circuit.h(0)
    circuit.cnot(0, 1)
    if params["swap"]:
        circuit.x(1)
    result = device.run(circuit, shots=0).result()
    probs = result.result_types[1].value
    print(f"{name}:")
    for state, prob in sorted(probs.items()):
        if prob > 1e-10:
            print(f"  |{state}> = {prob:.4f}")
    print()